# 최종 모델 평가
- 학습이 완료된 모델을 평가하는 코드입니다.
- teacher model(`GPT-4o-mini`)와 프롬프팅 엔지니어링을 활용하여, 전체 데이터에 대한 자동화된 평가를 수행합니다.
- 전체 데이터 중에서 20%의 데이터를 임의 추출하여, 사람이 직접 평가를 수행합니다.

In [85]:
from os import getenv
import os

import time

from dotenv import load_dotenv
from openai import OpenAI
import openai

import pandas as pd
import json
import ast

In [86]:
load_dotenv()
api_key = getenv("OPENAI_API_KEY")
client = openai.OpenAI(
    api_key = api_key
)

# step 값 확인하기!

In [87]:
step = 5
df = pd.read_excel(f"G:/내 드라이브/05_[공유파일]/Burnfit_dataset/eval_data_final.xlsx")

df.head()

,instruction,input,output
0,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 88.1, 'squat_1RM': 220.0, ...",허리가 안 좋으시군요. 그런 경우에는 데드리프트를 조절하는 것이 정말 중요합니다. ...
1,최근에 대회 준비를 시작했어. 강한 훈련이 필요해.,"{'성별': '남성', '몸무게': 105.23, 'squat_1RM': 230.0...",대회 준비를 시작하셨다니 정말 멋진 도전입니다! 강한 훈련이 필요하다는 목표에 맞춰...
2,하체 근력 향상에 포커스를 맞추고 싶어.,"{'성별': '남성', '몸무게': 89.3, 'squat_1RM': 165.0, ...",하체 근력 향상에 집중하고 싶으시다니 정말 멋진 목표입니다! 하체는 전신 운동에 큰...
3,전반적인 근력과 체력을 같이 키우고 싶어.,"{'성별': '남성', '몸무게': 81.8, 'squat_1RM': 195.0, ...",전반적인 근력과 체력을 키우고 싶으시다니 정말 멋진 목표입니다! 531 프로그램을 ...
4,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 110.0, 'squat_1RM': 222.3,...",허리가 안 좋으시군요. 그런 경우에는 데드리프트를 조절하는 것이 정말 중요합니다. ...


In [88]:
start_time = time.time()

inputs = []
instructions = []
gemma_outputs = []
outputs = []
model_name = "gpt-4o-mini"

for index, row in df.iterrows():
    try:
        input = row["input"]
        instruction = row["instruction"]

        output = row["output"]
        

        prompt =f"""
        ### 가이드라인 ###
        당신은 유능한 헬스 트레이너입니다.
        현재 531 운동 프로그램 루틴 추천에 관한 요청의 답변 정확도를 평가하고, 평가의 근거를 제시하는 업무를 수행하고 있습니다.
        당신의 답변은 ["평가의 근거", "평가"]의 형식을 갖추어야 합니다.
        또한, 문장 안에, 쌍따옴표(")를 사용하지마세요. ast.literal_eval() 함수를 활용하여, 파이썬 리스트 형태로 변환할 수 있어야 합니다.
        "평가의 근거"는 "평가"에 관한 당신의 의견을 제시한 문장의 형태입니다. 각 주차 별로 중량과 횟수가 제대로 계산 되었는지 확인해야 합니다. 쌍따옴표(")로 문장을 감싸야합니다.
        "평가의 근거"의 기준은 아래의 5가지 항목에 대해 평가를 수행해야 합니다.
            [1] 스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량(5 ~ 10% 오차범위 이내)과 횟수가 계산 되었다면 "적절"로 분류합니다.
            [2] 밀리터리프레스의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량(5 ~ 10% 오차범위 이내)과 횟수가 계산 되었다면 "적절"로 분류합니다.
            [3] 벤치프레스의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량(5 ~ 10% 오차범위 이내)과 횟수가 계산 되었다면 "적절"로 분류합니다.
            [4] 데드리프트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량(5 ~ 10% 오차범위 이내)과 횟수가 계산 되었다면 "적절"로 분류합니다.
            [5] 사용자에게 총 4주의 531 프로그램 루틴을 추천하였다면 "적절"로 분류합니다.
        "평가의 근거"는 다음과 같은 논리에 따라 작성되어야 합니다.
            [1] 사용자의 1RM에 따른 TM(실제 1RM의 90%)을 계산합니다.
            [2] 계산된 TM에 따라, 각 주차 별 추천 중량을 계산합니다.
            [3] 당신이 계산한 각 주차 별 추천 중량과 [답변]의 각 주차 별 추천 중량을 비교합니다.
            [4] 각 운동(스쿼트, 밀리터리프레스, 벤치프레스, 데드리프트) 별로 적절한 중량(5 ~ 10% 오차범위 이내)이 계산되었는지 확인합니다.
            [5] 이제 "평가의 근거" 기준에 따라 5가지 항목에 대한 평가를 수행합니다.
            [6] 평가 결과를 명시합니다.
        "평가"는 "적절"과 "부적절" 중에 하나의 값을 가집니다. 쌍따옴표(")로 문장을 감싸야합니다.
        "평가의 근거"의 5가지 항목 중에서 3가지 항목에 대하여 "적절"하다고 분류를 한 경우, "평가"를 "적절로 분류합니다.
        

        ### 예시1 ###
        [사용자 요청]
        하체 근력 향상에 포커스를 맞추고 싶어.

        [사용자의 중량]
        '성별': '남성', '몸무게': 89.3, 'squat_1RM': 165.0, 'press_1RM': 63.0, 'bench_press_1RM': 90.0, 'deadlift_1RM': 195.0
        
        [답변]
        하체 근력 향상에 집중하고 싶으시다니 정말 멋진 목표입니다! 하체는 전신 운동에 큰 영향을 미치기 때문에, 강한 하체를 만드는 것은 매우 중요합니다. 531 프로그램을 기반으로 주차별로 추천 중량을 안내해드릴게요.

        먼저, 당신의 1RM(들 수 있는 최대 무게)을 기준으로 TM(실제 1RM의 90%)을 계산해보겠습니다.

        - **스쿼트 TM**: 165kg × 0.9 = 148.5kg
        - **밀리터리 프레스 TM**: 63kg × 0.9 = 56.7kg
        - **벤치 프레스 TM**: 90kg × 0.9 = 81kg
        - **데드리프트 TM**: 195kg × 0.9 = 175.5kg

        이제 주차별로 추천 중량을 안내해드리겠습니다. 하체 근력 향상을 위해 스쿼트와 데드리프트의 중량을 약간 더 증량해드릴게요.

        ### 1주차 (TM의 65%, 75%, 85%):
        - **스쿼트**: 5회 95kg, 5회 110kg, 5회 이상 125kg
        - **밀리터리 프레스**: ....

        [당신의 답변]
        ["사용자의 1RM에 따라 계산한 TM(실제 1RM의 90%)은 다음과 같습니다. - **스쿼트 TM**: 165kg × 0.9 = 148.5kg ... 따라서, 5가지 항목 중에 4가지 항목을 적절하다고 분류했으므로, 해당 답변은 적절합니다.", "적절"]
        
        ### 예시2 ###
        [사용자 요청]
        하체 근력 향상에 포커스를 맞추고 싶어.

        [사용자의 중량]
        '성별': '남성', '몸무게': 89.3, 'squat_1RM': 165.0, 'press_1RM': 63.0, 'bench_press_1RM': 90.0, 'deadlift_1RM': 195.0
        
        [답변]
        하체 근력 향상에 집중하고 싶으시다니 정말 멋진 목표입니다! 하체는 전신 운동에 큰 영향을 미치기 때문에, 강한 하체를 만드는 것은 매우 중요합니다. 531 프로그램을 기반으로 주차별로 추천 중량을 안내해드릴게요.

        먼저, 당신의 1RM(들 수 있는 최대 무게)을 기준으로 TM(실제 1RM의 90%)을 계산해보겠습니다.

        - **스쿼트 TM**: 165kg × 0.9 = 148.5kg
        - **밀리터리 프레스 TM**: 63kg × 0.9 = 56.7kg
        - **벤치 프레스 TM**: 90kg × 0.9 = 81kg
        - **데드리프트 TM**: 195kg × 0.9 = 175.5kg

        이제 주차별로 추천 중량을 안내해드리겠습니다. 하체 근력 향상을 위해 스쿼트와 데드리프트의 중량을 약간 더 증량해드릴게요.

        ### 1주차 (TM의 65%, 75%, 85%):
        - **스쿼트**: 5회 95kg, 5회 110kg, 5회 이상 125kg
        - **밀리터리 프레스**: ....

        [당신의 답변]
        ["사용자의 1RM에 따라 계산한 TM(실제 1RM의 90%)은 다음과 같습니다. - **스쿼트 TM**: 165kg × 0.9 = 148.5kg ... 따라서, 5가지 항목 중에 2가지 항목을 적절하다고 분류했으므로, 해당 답변은 부적절합니다.", "부적절"]

        ### 요청 ###
        다음의 질문과 답변에 대해 평가해주세요.
        [사용자 요청]
        {instruction}

        [사용자의 중량]
        {input}
                
        [답변]
        {output}

        """
        response = client.chat.completions.create(
            model=f"{model_name}",
            messages=[{"role":"user","content":[{"type":"text","text":f"{prompt}"}]}],
            temperature=0.1,
            top_p=0.8
        )
        
        result = response.choices[0].message.content
        outputs.append(result)
        inputs.append(input)
        instructions.append(instruction)
        gemma_outputs.append(output)
        print(f"index : {index} | ✅ Good")
    except Exception as e:
        outputs.append(e)
        inputs.append(input)
        instructions.append(instruction)
        gemma_outputs.append(output)
        print(f"index : {index} | ❌ ERROR : {e}")

end_time = time.time()

elapsed_time = end_time - start_time
print(f"실행 시간: {elapsed_time:.2f}초")

index : 0 | ✅ Good
index : 1 | ✅ Good
index : 2 | ✅ Good
index : 3 | ✅ Good
index : 4 | ✅ Good
index : 5 | ✅ Good
index : 6 | ✅ Good
index : 7 | ✅ Good
index : 8 | ✅ Good
index : 9 | ✅ Good
index : 10 | ✅ Good
index : 11 | ✅ Good
index : 12 | ✅ Good
index : 13 | ✅ Good
index : 14 | ✅ Good
index : 15 | ✅ Good
index : 16 | ✅ Good
index : 17 | ✅ Good
index : 18 | ✅ Good
index : 19 | ✅ Good
index : 20 | ✅ Good
index : 21 | ✅ Good
index : 22 | ✅ Good
index : 23 | ✅ Good
index : 24 | ✅ Good
index : 25 | ✅ Good
index : 26 | ✅ Good
index : 27 | ✅ Good
index : 28 | ✅ Good
index : 29 | ✅ Good
index : 30 | ✅ Good
index : 31 | ✅ Good
index : 32 | ✅ Good
index : 33 | ✅ Good
index : 34 | ✅ Good
index : 35 | ✅ Good
index : 36 | ✅ Good
index : 37 | ✅ Good
index : 38 | ✅ Good
index : 39 | ✅ Good
index : 40 | ✅ Good
index : 41 | ✅ Good
index : 42 | ✅ Good
index : 43 | ✅ Good
index : 44 | ✅ Good
index : 45 | ✅ Good
index : 46 | ✅ Good
index : 47 | ✅ Good
index : 48 | ✅ Good
index : 49 | ✅ Good
index : 50

In [109]:
print(outputs[0])
print(outputs[1])
print(outputs[2])

["스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었으므로 적절로 분류합니다. 밀리터리프레스의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었으므로 적절로 분류합니다. 벤치프레스의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었으므로 적절로 분류합니다. 데드리프트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었으나, 허리를 고려하여 중량을 조절한 점이 반영되어야 하므로 적절로 분류합니다. 마지막으로, 총 4주의 531 프로그램 루틴을 추천하였으므로 적절로 분류합니다. 따라서, 5가지 항목 중에 4가지 항목을 적절하다고 분류했으므로, 해당 답변은 적절합니다.", "적절"]
["스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었는지 확인한 결과, 1주차에서 135kg, 155kg, 175kg은 TM의 65%, 75%, 85%에 해당하므로 적절합니다. 2주차에서 145kg, 165kg, 185kg도 적절하며, 3주차의 155kg, 175kg, 195kg도 적절합니다. 그러나 4주차의 85kg, 105kg, 125kg은 TM의 40%, 50%, 60%에 해당하지 않으므로 부적절합니다. 밀리터리프레스의 경우, 1주차 65kg, 75kg, 85kg은 적절하고, 2주차 70kg, 80kg, 90kg도 적절합니다. 3주차 75kg, 85kg, 95kg도 적절하나, 4주차의 40kg, 50kg, 60kg은 부적절합니다. 벤치프레스의 경우, 1주차 95kg, 110kg, 125kg은 적절하고, 2주차 105kg, 120kg, 135kg도 적절합니다. 3주차 110kg, 125kg, 140kg도 적절하나, 4주차의 60kg, 75kg, 90kg은 부적절합니다. 데드리프트의 경우, 1주차 135kg, 155kg, 175kg은 적절하고, 2주차 145kg, 165kg, 185kg도 적절합니다. 3주차 155kg, 175kg, 195kg도 적절하나, 4주차

In [124]:
eval = []
eval_str = []
index = 0

for output in outputs:
    try:
        output_ = ast.literal_eval(output)
        if len(output_) != 2:

            eval_str.append(output_[:-2])
            eval.append(output_[-1])
        else:
            eval_str.append(output_[0])
            eval.append(output_[1])
        print(f"index : {index} | ✅ Good")
        index += 1
    except Exception as e:
        print(len(output_))
        eval_str.append(index)
        eval.append(index)
        print(f"index : {index} | ❌ ERROR : {e}")

index : 0 | ✅ Good
index : 1 | ✅ Good
index : 2 | ✅ Good
index : 3 | ✅ Good
index : 4 | ✅ Good
index : 5 | ✅ Good
index : 6 | ✅ Good
index : 7 | ✅ Good
index : 8 | ✅ Good
index : 9 | ✅ Good
index : 10 | ✅ Good
index : 11 | ✅ Good
index : 12 | ✅ Good
index : 13 | ✅ Good
index : 14 | ✅ Good
index : 15 | ✅ Good
index : 16 | ✅ Good
index : 17 | ✅ Good
index : 18 | ✅ Good
index : 19 | ✅ Good
index : 20 | ✅ Good
index : 21 | ✅ Good
index : 22 | ✅ Good
index : 23 | ✅ Good
index : 24 | ✅ Good
index : 25 | ✅ Good
index : 26 | ✅ Good
index : 27 | ✅ Good
index : 28 | ✅ Good
index : 29 | ✅ Good
index : 30 | ✅ Good
index : 31 | ✅ Good
index : 32 | ✅ Good
index : 33 | ✅ Good
index : 34 | ✅ Good
index : 35 | ✅ Good
index : 36 | ✅ Good
index : 37 | ✅ Good
index : 38 | ✅ Good
index : 39 | ✅ Good
index : 40 | ✅ Good
index : 41 | ✅ Good
index : 42 | ✅ Good
index : 43 | ✅ Good
index : 44 | ✅ Good
index : 45 | ✅ Good
index : 46 | ✅ Good
index : 47 | ✅ Good
index : 48 | ✅ Good
index : 49 | ✅ Good
index : 50

In [125]:
len(eval_str), len(eval)

(100, 100)

In [126]:
df_result = pd.DataFrame()

df_result["input"] = inputs
df_result["instruction"] = instructions
df_result["output"] = gemma_outputs

df_result["eval"] = eval
df_result["eval_str"] = eval_str


In [127]:
df_result.head()

,input,instruction,output,eval,eval_str
0,"{'성별': '남성', '몸무게': 88.1, 'squat_1RM': 220.0, ...",요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,허리가 안 좋으시군요. 그런 경우에는 데드리프트를 조절하는 것이 정말 중요합니다. ...,적절,"스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었으므..."
1,"{'성별': '남성', '몸무게': 105.23, 'squat_1RM': 230.0...",최근에 대회 준비를 시작했어. 강한 훈련이 필요해.,대회 준비를 시작하셨다니 정말 멋진 도전입니다! 강한 훈련이 필요하다는 목표에 맞춰...,부적절,"스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었는지..."
2,"{'성별': '남성', '몸무게': 89.3, 'squat_1RM': 165.0, ...",하체 근력 향상에 포커스를 맞추고 싶어.,하체 근력 향상에 집중하고 싶으시다니 정말 멋진 목표입니다! 하체는 전신 운동에 큰...,사용자에게 총 4주의 531 프로그램 루틴을 추천하였으므로 적절합니다.,"[스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 추천 중량을 확인한 결과, 1주..."
3,"{'성별': '남성', '몸무게': 81.8, 'squat_1RM': 195.0, ...",전반적인 근력과 체력을 같이 키우고 싶어.,전반적인 근력과 체력을 키우고 싶으시다니 정말 멋진 목표입니다! 531 프로그램을 ...,적절,"스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 적절한 중량과 횟수가 계산되었으나..."
4,"{'성별': '남성', '몸무게': 110.0, 'squat_1RM': 222.3,...",요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,허리가 안 좋으시군요. 그런 경우에는 데드리프트를 조절하는 것이 정말 중요합니다. ...,적절,"스쿼트의 경우, 계산된 TM에 따라 각 주차 별로 추천 중량을 비교한 결과, 1주차..."


In [128]:
df_result.to_excel(f"G:/내 드라이브/05_[공유파일]/Burnfit_dataset/eval_result_final.xlsx", index=False)

In [129]:
df_result["eval"].value_counts()

eval
적절                                         60
부적절                                        38
사용자에게 총 4주의 531 프로그램 루틴을 추천하였으므로 적절합니다.     1
72                                          1
Name: count, dtype: int64

### 오류 수동 수정
- 총 요청 수 : 100
- 적절 : 61
- 부적절 : 39 (오류 1 포함)

In [132]:
sampled_df = df.sample(n=20, random_state=42)
sampled_df.head()

,instruction,input,output
83,요즘 벤치가 안 늘어서 고민이야. 벤치 집중 루틴이 포함된 4주 5/3/1 프로그램...,"{'성별': '남성', '몸무게': 91.1, 'squat_1RM': 245.0, ...",벤치 프레스에 집중하고 싶으시다니 정말 좋은 결정입니다! 531 프로그램을 통해 벤...
53,하체 근력 향상에 포커스를 맞추고 싶어.,"{'성별': '남성', '몸무게': 97.9, 'squat_1RM': 195.0, ...",하체 근력 향상에 집중하고 싶으시다니 정말 멋진 목표입니다! 하체는 전신 운동에 큰...
70,요즘 벤치가 안 늘어서 고민이야. 벤치 집중 루틴이 포함된 4주 5/3/1 프로그램...,"{'성별': '남성', '몸무게': 70.3, 'squat_1RM': 152.5, ...",벤치 프레스에 집중하고 싶으시다니 정말 좋은 결정입니다! 531 프로그램을 통해 벤...
45,하체 근력 향상에 포커스를 맞추고 싶어.,"{'성별': '남성', '몸무게': 82.4, 'squat_1RM': 200.0, ...",하체 근력 향상에 집중하고 싶으시다니 정말 멋진 목표입니다! 하체는 전신 운동에 큰...
44,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 91.8, 'squat_1RM': 205.0, ...",허리가 안 좋으시군요. 그런 경우에는 데드리프트를 조절하는 것이 정말 중요합니다. ...


In [ ]:
# 사람 직접 평가를 위한 파일 생성
sampled_df.to_excel(f"G:/내 드라이브/05_[공유파일]/Burnfit_dataset/eval_result_final(20).xlsx", index=False)